In [1]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

In [3]:
files = list(sorted(Path(".").glob("orders_week*.csv")))
print(f"Found files: {len(files)}")

Found files: 3


In [4]:
dataframe = []

for file in files:
    df = pd.read_csv(file)
    dataframe.append(df)
    print(file.name,':',len(df), "rows")

orders_week1.csv : 4 rows
orders_week2.csv : 4 rows
orders_week3.csv : 4 rows


In [5]:
orders = pd.concat(dataframe, ignore_index=True)

print(orders)
print("total rows: ", len(orders))

   order_id  order_date product  quantity  unit_price
0    ORD001  2026-10-01    สมุด         2          35
1    ORD002  2026-10-01  ปากกา          3          15
2    ORD003  2026-10-02   ยางลบ         1          10
3    ORD004  2026-10-02    สมุด         1          35
4    ORD005  2026-10-03   ปากกา         2          15
5    ORD002  2026-10-01   ปากกา         3          15
6    ORD006  2026-10-03    สมุด         3          35
7    ORD007  2026-10-04   ยางลบ         2          10
8    ORD008  2026-10-05   ปากกา         1          15
9    ORD009  2026-10-05    สมุด         2          35
10   ORD010  2026-10-06   ยางลบ         3          10
11   ORD011  2026-10-06   ปากกา         4          15
total rows:  12


In [6]:
unique_values = orders['product'].unique()
unique_values

<StringArray>
['สมุด', 'ปากกา ', 'ยางลบ', ' ปากกา', 'ปากกา']
Length: 5, dtype: str

In [7]:
orders['product'] = orders['product'].str.strip()


In [8]:
orders

,order_id,order_date,product,quantity,unit_price
0,ORD001,2026-10-01,สมุด,2,35
1,ORD002,2026-10-01,ปากกา,3,15
2,ORD003,2026-10-02,ยางลบ,1,10
3,ORD004,2026-10-02,สมุด,1,35
4,ORD005,2026-10-03,ปากกา,2,15
5,ORD002,2026-10-01,ปากกา,3,15
6,ORD006,2026-10-03,สมุด,3,35
7,ORD007,2026-10-04,ยางลบ,2,10
8,ORD008,2026-10-05,ปากกา,1,15
9,ORD009,2026-10-05,สมุด,2,35


In [9]:
duplicate_rows = orders[
    orders.duplicated(
        subset="order_id",
        keep=False
    )
]

conflicts = duplicate_rows.groupby("order_id").filter(
    lambda group: len(group.drop_duplicates()) > 1
)

if not conflicts.empty:
    print(conflicts)
    raise ValueError("Conflicting duplicate orders found")

orders = orders.drop_duplicates(
    subset="order_id",
    keep="first"
).copy()

print(duplicate_rows.sort_values(by=list(orders.columns)))

  order_id  order_date product  quantity  unit_price
1   ORD002  2026-10-01   ปากกา         3          15
5   ORD002  2026-10-01   ปากกา         3          15


In [10]:
orders['amount'] = orders['quantity'] * orders['unit_price']
orders

,order_id,order_date,product,quantity,unit_price,amount
0,ORD001,2026-10-01,สมุด,2,35,70
1,ORD002,2026-10-01,ปากกา,3,15,45
2,ORD003,2026-10-02,ยางลบ,1,10,10
3,ORD004,2026-10-02,สมุด,1,35,35
4,ORD005,2026-10-03,ปากกา,2,15,30
6,ORD006,2026-10-03,สมุด,3,35,105
7,ORD007,2026-10-04,ยางลบ,2,10,20
8,ORD008,2026-10-05,ปากกา,1,15,15
9,ORD009,2026-10-05,สมุด,2,35,70
10,ORD010,2026-10-06,ยางลบ,3,10,30


In [11]:
sales_by_product = (
    orders
    .groupby('product')
    .agg(
        total_quantity=('quantity', 'sum'),
        total_amount=('amount', 'sum')
    )
    .reset_index()
)

sales_by_product


,product,total_quantity,total_amount
0,ปากกา,10,150
1,ยางลบ,6,60
2,สมุด,8,280


In [12]:
sales_by_product.loc[len(sales_by_product)] = {
    'product': 'รวม',
    'total_quantity': sales_by_product['total_quantity'].sum(),
    'total_amount': sales_by_product['total_amount'].sum()
}

sales_by_product

,product,total_quantity,total_amount
0,ปากกา,10,150
1,ยางลบ,6,60
2,สมุด,8,280
3,รวม,24,490


In [13]:
orders.to_csv('cleaned_orders.csv', index=False, encoding='utf-8-sig')
sales_by_product.to_csv('sales_by_product.csv', index=False, encoding='utf-8-sig')